In [1]:
import torch
from torch.utils import data
from torchvision import transforms as T

from src.dataloaders.dataloader_for_CNN import mavDataLoader, mavDatasetCNN_3D, SequenceBatchSampler

device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print('Обучение на:', device, sep=' ')

transform = T.Compose([
    T.Resize((320, 192))
])
dataset = mavDatasetCNN_3D('datasets/euroc_mav', transform, device='cpu', batchsize=8, hidden_size=0, lst_of_datasets=['mav0_easy1']) # Сразу формируем все массивы на GPU

groups = dataset.batch_groups.copy()

train_size = int(0.8 * len(groups))
train_groups = groups[:train_size]
test_groups = groups[train_size:]


train_sampler = SequenceBatchSampler(train_groups)
test_sampler = SequenceBatchSampler(test_groups)

train_data = data.DataLoader(dataset, batch_sampler=train_sampler, num_workers=6, pin_memory=True)
test_data = data.DataLoader(dataset, batch_sampler=test_sampler, num_workers=6, pin_memory=True)

Обучение на: cuda


In [2]:
from src.geometry.RotationTorch import RotationTorch as RT 
from src.geometry.PoseTorch import PoseTorch as PT 

In [3]:
dt = iter(train_data)
x, y, T_m = next(dt)
x2, y2, T_m2 = next(dt)

In [4]:
y_p = PT.from_lie(y)
y_p[0]

PoseTorch(R=RotationTorch(_q=tensor([ 1.0000e+00, -5.1946e-04,  7.6956e-04, -3.0603e-04],
       dtype=torch.float64)), t=tensor([ 3.7289e-03, -9.8861e-07, -1.4966e-03], dtype=torch.float64))

In [5]:
torch.stack([y[0], y[1]], dim=0)

tensor([[ 3.7278e-03, -2.9076e-06, -1.4995e-03,  1.0389e-03, -1.5391e-03,
          6.1206e-04],
        [ 3.7516e-03,  1.6501e-06, -1.4949e-03,  1.0831e-03, -1.5670e-03,
          6.5200e-04]], dtype=torch.float64)

In [8]:
PT.stack([y_p[0], y_p[1]], dim=0).t.ndim

2

In [9]:
y_p[0].t.ndim

1

In [59]:
PT.from_lie(y2)

PoseTorch(R=RotationTorch(_q=tensor([[ 1.0000e+00, -5.9956e-04,  7.7205e-04, -3.8278e-04],
        [ 1.0000e+00, -5.7910e-04,  7.6669e-04, -3.8059e-04],
        [ 1.0000e+00, -5.5805e-04,  7.6156e-04, -3.6605e-04],
        [ 1.0000e+00, -5.3024e-04,  7.5999e-04, -3.3683e-04],
        [ 1.0000e+00, -4.9154e-04,  7.6109e-04, -3.0496e-04],
        [ 1.0000e+00, -4.4526e-04,  7.5535e-04, -2.6417e-04],
        [ 1.0000e+00, -3.9282e-04,  7.5998e-04, -2.1901e-04],
        [ 1.0000e+00, -3.4293e-04,  7.6617e-04, -1.8041e-04]],
       dtype=torch.float64)), t=tensor([[ 3.7582e-03,  2.4219e-05, -1.4690e-03],
        [ 3.7510e-03,  1.9028e-05, -1.4642e-03],
        [ 3.7442e-03,  1.3138e-05, -1.4611e-03],
        [ 3.7403e-03,  2.8293e-06, -1.4563e-03],
        [ 3.7382e-03, -7.5668e-06, -1.4500e-03],
        [ 3.7398e-03, -1.8376e-05, -1.4453e-03],
        [ 3.7426e-03, -2.9744e-05, -1.4378e-03],
        [ 3.7454e-03, -3.9383e-05, -1.4305e-03]], dtype=torch.float64))

In [53]:
y_b = torch.stack([y, y2])
T_m_b = torch.stack([T_m, T_m2])

T_M = PT.from_lie(T_m)
Y = PT.from_lie(y_b)

result = T_M * Y

result[0]

PoseTorch(R=RotationTorch(_q=tensor([[ 0.5346, -0.1530, -0.8270, -0.0829],
        [ 0.5346, -0.1530, -0.8270, -0.0830]], dtype=torch.float64)), t=tensor([[ 4.6882, -1.7868,  0.7873],
        [ 4.6882, -1.7867,  0.7874]], dtype=torch.float64))

In [65]:
PT.from_lie(T_m[1]) * PT.from_lie(y[1]), PT.from_lie(T_m[4]) * PT.from_lie(y2[4])

(PoseTorch(R=RotationTorch(_q=tensor([ 0.5352, -0.1529, -0.8266, -0.0836], dtype=torch.float64)), t=tensor([ 4.6880, -1.7866,  0.7914], dtype=torch.float64)),
 PoseTorch(R=RotationTorch(_q=tensor([ 0.5368, -0.1528, -0.8253, -0.0859], dtype=torch.float64)), t=tensor([ 4.6876, -1.7861,  0.8035], dtype=torch.float64)))

In [54]:
T_M[0] * Y[0]

PoseTorch(R=RotationTorch(_q=tensor([[ 0.5346, -0.1530, -0.8270, -0.0829],
        [ 0.5346, -0.1530, -0.8270, -0.0830]], dtype=torch.float64)), t=tensor([[ 4.6882, -1.7868,  0.7873],
        [ 4.6882, -1.7867,  0.7874]], dtype=torch.float64))

In [49]:
T_M[1]

PoseTorch(R=RotationTorch(_q=tensor([[ 0.5346, -0.1530, -0.8270, -0.0829],
        [ 0.5388, -0.1526, -0.8237, -0.0894]], dtype=torch.float64)), t=tensor([[ 4.6882, -1.7868,  0.7873],
        [ 4.6870, -1.7854,  0.8197]], dtype=torch.float64))

In [5]:
(PT.from_lie(T_m) * PT.from_lie(y)).as_lie()

tensor([[ 1.9798, -1.7495,  5.4164,  0.3645,  1.9703,  0.1974],
        [ 1.9773, -1.7527,  5.4154,  0.3643,  1.9689,  0.1991],
        [ 1.9746, -1.7561,  5.4144,  0.3641,  1.9674,  0.2010],
        [ 1.9719, -1.7598,  5.4135,  0.3639,  1.9660,  0.2029],
        [ 1.9691, -1.7634,  5.4125,  0.3636,  1.9645,  0.2048],
        [ 1.9663, -1.7670,  5.4116,  0.3635,  1.9631,  0.2068],
        [ 1.9634, -1.7706,  5.4107,  0.3633,  1.9617,  0.2087],
        [ 1.9606, -1.7742,  5.4098,  0.3631,  1.9603,  0.2106]],
       dtype=torch.float64)

In [6]:
T_pt = PT.from_lie(T_m)
T_pt

PoseTorch(R=RotationTorch(_q=tensor([[ 0.5341, -0.1530, -0.8274, -0.0822],
        [ 0.5346, -0.1530, -0.8270, -0.0829],
        [ 0.5352, -0.1529, -0.8266, -0.0836],
        [ 0.5357, -0.1529, -0.8261, -0.0844],
        [ 0.5362, -0.1528, -0.8257, -0.0852],
        [ 0.5368, -0.1528, -0.8253, -0.0860],
        [ 0.5373, -0.1527, -0.8249, -0.0869],
        [ 0.5378, -0.1527, -0.8245, -0.0877]], dtype=torch.float64)), t=tensor([[ 4.6883, -1.7869,  0.7833],
        [ 4.6882, -1.7868,  0.7873],
        [ 4.6880, -1.7866,  0.7914],
        [ 4.6879, -1.7864,  0.7954],
        [ 4.6877, -1.7862,  0.7995],
        [ 4.6876, -1.7861,  0.8035],
        [ 4.6874, -1.7859,  0.8076],
        [ 4.6873, -1.7857,  0.8116]], dtype=torch.float64))

In [7]:
T_pt.R[0]

RotationTorch(_q=tensor([ 0.5341, -0.1530, -0.8274, -0.0822], dtype=torch.float64))

In [13]:
y_pt = PT.from_lie(y)
T_pt[0] * y_pt[0]

PoseTorch(R=RotationTorch(_q=tensor([ 0.5346, -0.1530, -0.8270, -0.0829], dtype=torch.float64)), t=tensor([ 4.6882, -1.7868,  0.7873], dtype=torch.float64))

In [22]:
T_pt[1:3]

PoseTorch(R=RotationTorch(_q=tensor([[ 0.5346, -0.1530, -0.8270, -0.0829],
        [ 0.5352, -0.1529, -0.8266, -0.0836]], dtype=torch.float64)), t=tensor([[ 4.6882, -1.7868,  0.7873],
        [ 4.6880, -1.7866,  0.7914]], dtype=torch.float64))

In [ ]:
shape = T_pt.t.shape[:-1]
z = torch.zeros(*shape, 4)
z[..., 0] = 1

RT.from_quat(z)

RotationTorch(_q=tensor([[1., 0., 0., 0.],
        [1., 0., 0., 0.],
        [1., 0., 0., 0.],
        [1., 0., 0., 0.],
        [1., 0., 0., 0.],
        [1., 0., 0., 0.],
        [1., 0., 0., 0.],
        [1., 0., 0., 0.]]))

In [20]:
y_s = y.unsqueeze(0)
PT.from_lie(y_s)[1]

PoseTorch(R=RotationTorch(_q=tensor([[ 1.0000e+00, -5.4155e-04,  7.8350e-04, -3.2600e-04]],
       dtype=torch.float64)), t=tensor([[ 3.7527e-03,  3.6819e-06, -1.4919e-03]], dtype=torch.float64))